# 03 — Time-series diagnostics and model eligibility

This notebook is the first formal econometric diagnostics stage. Run notebooks 01 and 02 first. The notebook reads the cleaned monthly data and the original quarterly GDP data; it does not use interpolated GDP.

The aim is **not** to select the model that gives the smallest p-values. It is to document the time-series properties of the candidate variables and establish which model classes are eligible under the research protocol.

Main steps:
1. Audit sample continuity and large changes before testing.
2. Test levels and first differences using ADF, Phillips–Perron (PP) and KPSS.
3. Run a one-break unit-root sensitivity test on the level series and record the estimated break date.
4. Keep quarterly GDP at quarterly frequency.
5. Provide a gated, optional lag-order exercise for a candidate monthly VAR.

The deterministic terms are tested as alternative specifications. A constant and a constant-plus-trend are not interchangeable; the final choice must be justified by the series plot and economic meaning.

In [ ]:
from pathlib import Path
import sys
import platform
import arch
import statsmodels

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.api import VAR

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

# Resolve the project root whether Jupyter starts at the repository root or in notebooks/.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(
    (path for path in candidates if (path / "data" / "processed" / "ghana_banking_monthly_clean.csv").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Could not find data/processed/ghana_banking_monthly_clean.csv. "
        "Run notebook 01_data_cleaning.ipynb first."
    )

sys.path.insert(0, str(ROOT / "src"))
from ghana_banking.econometrics import run_unit_root_suite, run_zivot_andrews_test

PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "figures"
PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print(f"Repository: {ROOT}")
environment = pd.DataFrame([{
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "statsmodels": statsmodels.__version__,
    "arch": arch.__version__,
}])
display(environment)

## 1. Load and audit the monthly sample

The primary monthly analysis excludes GDP. Confirm that the date index is monthly, unique and ordered. Any missing calendar month is reported; it is not silently inserted or imputed.

In [ ]:
monthly_path = PROCESSED / "ghana_banking_monthly_clean.csv"
monthly = pd.read_csv(monthly_path, parse_dates=["Date"], index_col="Date")
monthly = monthly.sort_index().apply(pd.to_numeric, errors="coerce")

if monthly.empty:
    raise ValueError("The cleaned monthly dataset is empty.")
if monthly.index.has_duplicates:
    raise ValueError("Duplicate monthly dates found.")
if not monthly.index.is_monotonic_increasing:
    raise ValueError("Monthly dates are not in chronological order.")

required_columns = [
    "Monetary Policy Rate (%)",
    "Consumer Price Index, All Items",
    "USD Exchange Rate, monthly averages",
    "Total Liquidity (M2+)",
    "Non Performing Loan Ratio",
]
missing_columns = sorted(set(required_columns) - set(monthly.columns))
if missing_columns:
    raise ValueError(f"Missing required monthly columns: {missing_columns}")

expected_months = pd.date_range(monthly.index.min(), monthly.index.max(), freq="MS")
missing_months = expected_months.difference(monthly.index)

sample_audit = pd.DataFrame(
    {
        "first_valid": monthly[required_columns].apply(lambda col: col.first_valid_index()),
        "last_valid": monthly[required_columns].apply(lambda col: col.last_valid_index()),
        "observed": monthly[required_columns].notna().sum(),
        "missing": monthly[required_columns].isna().sum(),
        "missing_pct": 100 * monthly[required_columns].isna().mean(),
    }
)
display(sample_audit)

print(f"Monthly source coverage: {monthly.index.min():%Y-%m} to {monthly.index.max():%Y-%m}")
print(f"Rows in cleaned file: {len(monthly):,}")
print(f"Missing calendar months: {len(missing_months)}")
if len(missing_months):
    display(pd.DataFrame({"missing_month": missing_months}))

## 2. Construct the core monthly series

Positive price and quantity indices are analysed in logarithms. The NPL ratio and policy rate remain in their published percentage units. First differences of the NPL ratio and policy rate are therefore **percentage-point changes**; first differences of logged CPI, the exchange rate and M2+ are approximate monthly percentage changes.

M2+ receives a separate data-quality check because unusually large historical movements are visible in the source file. Such observations are flagged for source verification, not automatically deleted or winsorised.

In [ ]:
positive_columns = [
    "Consumer Price Index, All Items",
    "USD Exchange Rate, monthly averages",
    "Total Liquidity (M2+)",
]
for column in positive_columns:
    observed = monthly[column].dropna()
    if observed.empty or (observed <= 0).any():
        raise ValueError(f"{column} must contain positive observed values for log transformations.")

core_levels = pd.DataFrame(index=monthly.index)
core_levels["NPL ratio (%)"] = monthly["Non Performing Loan Ratio"]
core_levels["Policy rate (%)"] = monthly["Monetary Policy Rate (%)"]
core_levels["log CPI"] = np.log(monthly["Consumer Price Index, All Items"])
core_levels["log GHS per USD"] = np.log(monthly["USD Exchange Rate, monthly averages"])
core_levels["log M2+"] = np.log(monthly["Total Liquidity (M2+)"])

# Keep a named transformation table for interpretation and later modelling.
candidate_changes = pd.DataFrame(index=monthly.index)
candidate_changes["NPL change (percentage points)"] = core_levels["NPL ratio (%)"].diff()
candidate_changes["Policy-rate change (percentage points)"] = core_levels["Policy rate (%)"].diff()
candidate_changes["Inflation (monthly log change, %)"] = 100 * core_levels["log CPI"].diff()
candidate_changes["Cedi depreciation (monthly log change, %)"] = 100 * core_levels["log GHS per USD"].diff()
candidate_changes["M2+ growth (monthly log change, %)"] = 100 * core_levels["log M2+"].diff()

display(candidate_changes.describe().T)

## 3. Flag unusually large changes for source review

The robust z-score uses the median and median absolute deviation (MAD). A threshold of five is a screening rule only. The result is a review list, not an instruction to remove data. Check flagged dates against the original Bank of Ghana or GSS release and the relevant series definition before deciding how to treat them.

In [ ]:
def robust_change_flags(changes, threshold=5.0):
    """Return large-change flags using a median/MAD robust z-score."""
    rows = []
    for column in changes.columns:
        values = changes[column].replace([np.inf, -np.inf], np.nan).dropna()
        if values.empty:
            continue
        median = values.median()
        mad = (values - median).abs().median()
        if not np.isfinite(mad) or mad == 0:
            continue
        robust_z = 0.67448975 * (values - median) / mad
        flagged = robust_z.loc[robust_z.abs() >= threshold]
        for date, score in flagged.items():
            rows.append(
                {
                    "series": column,
                    "date": date,
                    "observed_change": values.loc[date],
                    "robust_z_score": score,
                }
            )
    return pd.DataFrame(rows, columns=["series", "date", "observed_change", "robust_z_score"])

change_flags = robust_change_flags(candidate_changes, threshold=5.0)
display(change_flags)

change_flags.to_csv(PROCESSED / "monthly_large_change_flags.csv", index=False)
print("Saved screening output: data/processed/monthly_large_change_flags.csv")
print("No observations have been removed or modified.")

## 4. ADF, PP and KPSS tests

The ADF and PP null hypotheses are that the series contains a unit root. The KPSS null is stationarity around the specified deterministic component. Agreement across tests is more informative than any one p-value, but no automatic integration-order label is assigned.

For levels, both a constant-only and a constant-plus-linear-trend specification are reported. For first differences, the constant-only specification is reported. The trend specification must ultimately be selected using the plot and economic rationale. ADF lag selection uses BIC; PP uses its documented automatic Newey–West bandwidth rule; KPSS uses automatic bandwidth selection.

The p-values are unadjusted, and the tests have finite-sample limitations. Treat conflicting or borderline evidence as inconclusive rather than forcing an I(0)/I(1) classification.

In [ ]:
def run_unit_root_suite_for_notebook(series_name, level_series, frequency):
    """Run the package diagnostic engine on levels and first differences."""
    records = []
    transformations = [
        ("level", level_series),
        ("first difference", level_series.diff()),
    ]

    for transformation, raw_series in transformations:
        values = raw_series.dropna()
        if len(values) < 40 or values.nunique() < 2:
            raise ValueError(
                f"Insufficient usable variation for {series_name} ({transformation}): "
                f"{len(values)} observations."
            )

        trend_specs = ["c", "ct"] if transformation == "level" else ["c"]
        for trend in trend_specs:
            test_results = run_unit_root_suite(
                series=values,
                series_name=series_name,
                deterministic=trend,
                adf_method="bic",
                frequency=frequency,
                transformation=transformation,
            )
            for result in test_results:
                records.append(
                    {
                        "frequency": result.frequency,
                        "sample_start": result.sample_start,
                        "sample_end": result.sample_end,
                        "input_observations": result.observations,
                        "series": result.series_name,
                        "transformation": transformation,
                        "deterministic_terms": result.deterministic,
                        "test": "PP" if result.test == "Phillips-Perron" else result.test,
                        "null_hypothesis": (
                            "stationarity" if result.test == "KPSS" else "unit root"
                        ),
                        "statistic": result.statistic,
                        "p_value": result.pvalue,
                        "lags_or_bandwidth": result.selected_lags,
                        "effective_nobs": result.effective_observations,
                        "critical_value_5pct": result.critical_values.get("5%"),
                        "lag_method": result.lag_method,
                    }
                )
    return records


monthly_unit_root_rows = []
for series_name in core_levels.columns:
    monthly_unit_root_rows.extend(
        run_unit_root_suite_for_notebook(
            series_name=series_name,
            level_series=core_levels[series_name],
            frequency="monthly",
        )
    )

monthly_unit_root_results = pd.DataFrame(monthly_unit_root_rows)
display(
    monthly_unit_root_results.sort_values(
        ["series", "transformation", "deterministic_terms", "test"]
    ).reset_index(drop=True)
)

monthly_unit_root_results.to_csv(
    PROCESSED / "monthly_unit_root_tests.csv", index=False
)
print("Saved: data/processed/monthly_unit_root_tests.csv")

### How to read the test combination

| ADF / PP | KPSS | Initial interpretation |
|---|---|---|
| Reject unit-root null | Do not reject stationarity null | Evidence is broadly consistent with stationarity under the tested deterministic terms. |
| Do not reject unit-root null | Reject stationarity null | Evidence is broadly consistent with a unit root under the tested deterministic terms. |
| Both nulls rejected | Conflicting evidence; possible breaks, nonlinearities, deterministic-term mismatch or finite-sample issues. |
| Neither null rejected | Tests are inconclusive; persistence and low power may be relevant. |

This table is a reading aid, not a mechanical classifier. Inspect plots, deterministic terms, lag choices, sample dates and break sensitivity before assigning an integration order.

## 5. One-break unit-root sensitivity

The Zivot–Andrews test allows one unknown break in the deterministic component under the unit-root null. It is included as a sensitivity check, not a universal fix for structural change. The test does not replace multiple-break analysis, parameter-stability tests or an economic explanation of candidate break periods.

The baseline specification allows a break in both the intercept and linear trend (`ct`) and uses BIC lag selection with a maximum of 12 lags. The reusable project function records the estimated break date, selected lag, critical values and sample metadata. The date is an estimated statistical breakpoint, not automatically the date of a known economic event.

In [ ]:
za_rows = []
for series_name in core_levels.columns:
    values = core_levels[series_name].dropna()
    za_result = run_zivot_andrews_test(
        series=values,
        series_name=series_name,
        regression="ct",
        max_lags=12,
        autolag="BIC",
        trim=0.15,
        frequency="monthly",
        transformation="level",
    )
    za_rows.append(za_result.to_dict())

za_results = pd.DataFrame(za_rows).set_index("series_name")
display(za_results)
za_results.to_csv(PROCESSED / "monthly_zivot_andrews_tests.csv")
print("Saved: data/processed/monthly_zivot_andrews_tests.csv")

## 6. Quarterly real GDP: separate frequency, separate test

Quarterly GDP is loaded from the original cleaned quarterly file. It is not joined to the monthly sample or interpolated. The log level and quarterly log growth are tested separately. Because the quarterly sample is shorter, results should be interpreted with particular caution.

In [ ]:
gdp_path = PROCESSED / "ghana_real_gdp_quarterly_clean.csv"
gdp = pd.read_csv(gdp_path)
if not {"Quarter", "GDP_Real"}.issubset(gdp.columns):
    raise ValueError("Quarterly GDP file must contain Quarter and GDP_Real.")

gdp["Quarter"] = pd.PeriodIndex(gdp["Quarter"], freq="Q")
gdp["GDP_Real"] = pd.to_numeric(gdp["GDP_Real"], errors="coerce")
gdp = gdp.dropna(subset=["Quarter", "GDP_Real"]).sort_values("Quarter")
gdp = gdp.drop_duplicates("Quarter").set_index("Quarter")

if (gdp["GDP_Real"] <= 0).any():
    raise ValueError("Quarterly real GDP must be positive before taking logs.")

log_gdp = np.log(gdp["GDP_Real"])
quarterly_unit_root_results = pd.DataFrame(
    run_unit_root_suite_for_notebook(
        series_name="log real GDP (constant 2013 prices)",
        level_series=log_gdp,
        frequency="quarterly",
    )
)
display(quarterly_unit_root_results)

quarterly_unit_root_results.to_csv(
    PROCESSED / "quarterly_gdp_unit_root_tests.csv", index=False
)
print(f"Quarterly GDP sample: {gdp.index.min()} to {gdp.index.max()}")
print(f"Saved: data/processed/quarterly_gdp_unit_root_tests.csv")

## 7. Candidate VAR lag-order exercise (deliberately gated)

Do not execute lag selection until you have reviewed the unit-root results and decided that the candidate transformed variables are appropriate for a stationary VAR. The flag is False by default.

If enabled, the example uses four variables and a maximum of six monthly lags. It reports information-criterion choices for review; it does not automatically designate the AIC, BIC or HQIC winner as the final model. The final choice must account for parameter count, sample size, residual diagnostics and economic rationale.

In [ ]:
RUN_CANDIDATE_VAR_LAG_SELECTION = False

if RUN_CANDIDATE_VAR_LAG_SELECTION:
    candidate_var = pd.DataFrame(
        {
            "NPL_change_pp": candidate_changes["NPL change (percentage points)"],
            "Policy_rate_change_pp": candidate_changes["Policy-rate change (percentage points)"],
            "Inflation_mom_pct": candidate_changes["Inflation (monthly log change, %)"],
            "Cedi_depreciation_mom_pct": candidate_changes[
                "Cedi depreciation (monthly log change, %)"
            ],
        },
        index=monthly.index,
    ).dropna()

    n_obs, n_variables = candidate_var.shape
    max_estimable_lags = (n_obs - n_variables - 1) // (n_variables + 1)
    max_lags_to_check = min(6, max_estimable_lags)

    if max_lags_to_check < 1:
        raise ValueError("The complete-case sample is too small for a VAR lag-order exercise.")

    lag_selection = VAR(candidate_var).select_order(maxlags=max_lags_to_check)
    lag_orders = pd.DataFrame([lag_selection.selected_orders])
    display(lag_orders)
    display(pd.DataFrame(lag_selection.ics))

    lag_orders.to_csv(PROCESSED / "candidate_var_selected_lags.csv", index=False)
    pd.DataFrame(lag_selection.ics).to_csv(
        PROCESSED / "candidate_var_information_criteria.csv"
    )
    print(
        f"Candidate VAR: {n_obs} observations, {n_variables} variables, "
        f"maximum lags checked = {max_lags_to_check}."
    )
else:
    print(
        "Lag selection not run. Review the unit-root and transformation evidence, "
        "then set RUN_CANDIDATE_VAR_LAG_SELECTION = True only if a stationary VAR is eligible."
    )

## 8. Diagnostic plots and files

Inspect the level and transformed series before deciding whether a constant, trend or break specification is economically plausible. The vertical scales differ across panels.

Files written by this notebook:
- monthly_unit_root_tests.csv
- monthly_zivot_andrews_tests.csv (including estimated break dates and critical values)
- monthly_large_change_flags.csv
- quarterly_gdp_unit_root_tests.csv
- optional candidate_var_selected_lags.csv and candidate_var_information_criteria.csv

In [ ]:
plot_series = {
    "NPL ratio (%)": core_levels["NPL ratio (%)"],
    "Policy rate (%)": core_levels["Policy rate (%)"],
    "log CPI": core_levels["log CPI"],
    "log GHS per USD": core_levels["log GHS per USD"],
    "log M2+": core_levels["log M2+"],
}
fig, axes = plt.subplots(len(plot_series), 1, figsize=(13, 13), sharex=True)
for axis, (label, values) in zip(axes, plot_series.items()):
    axis.plot(values.index, values, linewidth=1.3)
    axis.set_ylabel(label)
    axis.grid(alpha=0.25)
axes[-1].set_xlabel("Month")
fig.suptitle("Candidate monthly series for time-series modelling", y=1.002)
fig.tight_layout()
fig.savefig(FIGURES / "monthly_candidate_series_for_modelling.png", dpi=160, bbox_inches="tight")
plt.show()

print("Diagnostic outputs saved under data/processed/.")
print("No model has been estimated and no integration order has been assigned automatically.")

## 9. Decision record before model estimation

Complete this decision record after reviewing the exported tables and plots. Do not fill it with conclusions that the test evidence does not support.

| Variable | Deterministic terms justified? | ADF / PP evidence | KPSS evidence | Break sensitivity | Working integration assessment | Eligible model treatment |
|---|---|---|---|---|---|---|
| NPL ratio | To be assessed | To be assessed | To be assessed | To be assessed | Pending | Pending |
| Policy rate | To be assessed | To be assessed | To be assessed | To be assessed | Pending | Pending |
| log CPI | To be assessed | To be assessed | To be assessed | To be assessed | Pending | Pending |
| log GHS per USD | To be assessed | To be assessed | To be assessed | To be assessed | Pending | Pending |
| log M2+ | To be assessed | To be assessed | To be assessed | To be assessed | Pending | Pending |
| log real GDP (quarterly) | To be assessed | To be assessed | To be assessed | To be assessed | Pending | Pending |

### Model eligibility rules

- ARDL bounds testing is considered only where no included variable is I(2) and the deterministic case and sample support the test.
- VECM is considered where the included system variables are I(1) and cointegration rank evidence supports a cointegrating relationship.
- Stationary VAR is considered only after the chosen transformations produce a defensible stationary system.
- If evidence is conflicting, the order of integration is uncertain or structural breaks dominate the results, report that uncertainty and revisit the specification instead of forcing a model choice.

### References for test implementations

- ADF: Dickey and Fuller (1979), https://doi.org/10.1080/01621459.1979.10482531
- PP: Phillips and Perron (1988), https://doi.org/10.1093/biomet/75.2.335
- KPSS: Kwiatkowski, Phillips, Schmidt and Shin (1992), https://doi.org/10.1016/0304-4076(92)90104-Y
- Zivot–Andrews: Zivot and Andrews (1992), https://doi.org/10.1080/07350015.1992.10509904

The Phillips–Perron routine is provided by `arch`; the reusable Zivot–Andrews wrapper uses `statsmodels.tsa.stattools.zivot_andrews` so the estimated break index/date can be retained. See the statsmodels time-series documentation for implementation details.